In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Оцінка часу використання: одна хвилина на процесорі Eagle (ПРИМІТКА: Це лише оцінка. Фактичний час виконання може відрізнятися.)*

## Передумови

Розрізання схем — це загальний термін, який охоплює різні методи розбиття схеми на кілька менших підсхем, що включають меншу кількість вентилів та/або кубітів. Кожна з підсхем може виконуватися незалежно, а кінцевий результат отримується шляхом класичної постобробки результатів кожної підсхеми. Ця техніка доступна у [додатку Qiskit для розрізання схем](https://qiskit.github.io/qiskit-addon-cutting/index.html), детальне пояснення техніки наведено у [документації](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) разом з іншими [вступними матеріалами](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Цей блокнот розглядає метод під назвою <b>розрізання дротів</b>, де схема розбивається вздовж дроту [\[1\], \[2\]](#references). Зауважте, що розбиття є простим у класичних схемах, оскільки результат у точці розбиття можна визначити детерміновано, і він дорівнює 0 або 1. Однак стан кубіта в точці розрізу, як правило, є змішаним станом. Тому кожну підсхему потрібно вимірювати кілька разів у різних базисах (зазвичай томографічно повний набір базисів, такий як базис Паулі [\[3\], \[4\]](#references) і відповідно готувати у власному стані. На наведеному нижче малюнку (<i>надано: PhD Thesis, Ritajit Majumdar</i>) показано приклад розрізання дротів для 4-кубітного стану GHZ на три підсхеми. Тут $M_j$ позначає набір базисів (зазвичай Паулі X, Y і Z), а $P_i$ позначає набір власних станів (зазвичай $|0\rangle$, $|1\rangle$, $|+\rangle$ і $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Оскільки кожна підсхема має меншу кількість кубітів та/або вентилів, очікується, що вони будуть менш схильні до шуму. Цей блокнот показує приклад, де цей метод може використовуватися для ефективного придушення шуму в системі.

## Вимоги

Перед початком цього навчального посібника переконайтеся, що у Вас встановлено наступне:

- Qiskit SDK v2.0 або новіша версія з підтримкою [візуалізації](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 або новіша версія ( `pip install qiskit-ibm-runtime` )
- Додаток Qiskit для розрізання схем v0.9.0 або новіша версія (`pip install qiskit-addon-cutting`)

Ми розглянемо схему багаточастинкової локалізації (MBL) для цього блокноту. Схема MBL є апаратно-ефективною схемою і параметризована двома параметрами $\theta$ і $\vec{\phi}$. Коли $\theta$ встановлено на $0$, а початковий стан підготовлено у $|0\rangle$ для всіх кубітів, ідеальне очікуване значення $\langle Z_i \rangle$ дорівнює $+1$ для кожної позиції кубіта $i$ незалежно від значень $\vec{\phi}$. Ви можете ознайомитися з додатковими деталями про схеми MBL у <a href="https://arxiv.org/abs/2307.07552">цій статті</a>.

## Налаштування

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Частина I. Приклад малого масштабу

### Крок 1: Зіставлення класичних вхідних даних з квантовою задачею

Спочатку ми побудуємо шаблонну схему без конкретних значень параметрів. Ми також надаємо заповнювачі, які називаються `CutWire`, для позначення позиції розрізів. Для прикладу малого масштабу ми розглянемо 10-кубітну схему MBL.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Тепер ми позначимо схему для розрізання, вставивши відповідні **CutWire** для створення двох приблизно рівних розрізів. Ми встановимо `use_cut=True` у функції та дозволимо їй позначити після $\frac{n}{2}$ кубітів, де $n$ — кількість кубітів у вихідній схемі.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Крок 2: Оптимізація задачі для виконання на квантовому обладнанні
Далі ми розріжемо схему на дві менші підсхеми. Для цього прикладу ми обмежимося лише 2 підсхемами. Для цього ми використовуємо <a href="https://qiskit.github.io/qiskit-addon-cutting/">Додаток Qiskit: Розрізання схем</a>.
#### Розрізання схеми на менші підсхеми
Розрізання дроту в точці збільшує кількість кубітів на один. Окрім вихідного кубіта, тепер є додатковий кубіт як заповнювач для схеми після розрізання. Наступне зображення дає уявлення:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Цей додаток використовує функцію `cut_wires` для обліку додаткових кубітів, що виникають внаслідок розрізання.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Створення та розширення спостережуваних величин
Тепер ми побудуємо спостережувану величину $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Оскільки ідеальний результат $\langle Z_i \rangle$ для кожного $i$ дорівнює $+1$, ідеальний результат $M_z$ також дорівнює $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Однак зауважте, що кількість кубітів у схемі збільшилася після вставки віртуальних 2-кубітних операцій `Move` після розрізання. Тому нам також потрібно розширити спостережувані величини, вставивши одиниці для відповідності поточній схемі.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Давайте візуалізуємо підсхеми

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Спостережувані величини також були розбиті для відповідності підсхемам

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

Зауважте, що кожна підсхема призводить до певної кількості вибірок. Реконструкція враховує результат кожної з цих вибірок. Кожна з цих вибірок називається `subexperiment`.
Розширення спостережуваної величини за допомогою операції `Move` вимагає структури даних `PauliList`. Ми також можемо створити спостережувану величину $M_z$ у більш загальній структурі даних `SparsePauliOp`, що буде корисно пізніше під час реконструкції підекспериментів.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Розглянемо два приклади, де розрізані кубіти вимірюються у двох різних базисах. Спочатку вони вимірюються у звичайному базисі Z, а потім у базисі X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Транспіляція кожного підексперименту

Наразі нам потрібно транспілювати наші схеми перед відправленням їх на виконання. Тому ми спочатку транспілюємо кожну схему в підекспериментах.